# Notebook 01 — Download the datasets (HDFS + BGL)

LOG6309E group assignment: replication of **RQ1** from Wu, Li & Khomh, *"On the Effectiveness
of Log Representation for Log-based Anomaly Detection"* (EMSE 2023).

This notebook fetches the two datasets used by our pipeline from **Loghub** (Zenodo record
8196385), arranges them in the layout expected by `src/` and `notebooks/02_replicate_rq1.ipynb`,
and validates basic statistics against the paper:

| Dataset | Archive | Expected content |
|---|---|---|
| HDFS | `HDFS_v1.zip` → `HDFS.log` + `anomaly_label.csv` | 11,175,629 lines; 575,061 blocks, 16,838 anomalous (2.9%) |
| BGL  | `BGL.zip` → `BGL.log` | 4,747,963 lines; 348,460 alert lines (7.3%) |

Everything is stored on Google Drive (≈ 2.2 GB) so notebook 02 can reuse it without
re-downloading. Runtime ≈ 10–15 minutes. Re-runs skip what is already on Drive.

In [ ]:
import glob
import json
import os
import shutil
import sys
import tarfile
import time
import zipfile
import urllib.request

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/LOG6309E'   # persistent storage across Colab sessions
    WORK_DIR = '/content/work'                       # fast VM disk for download + extraction
else:
    # local run: everything under notebooks/cache (gitignored)
    WORK_DIR = os.path.join(os.getcwd(), 'cache')
    DRIVE_ROOT = WORK_DIR

for d in (WORK_DIR, os.path.join(DRIVE_ROOT, 'data')):
    os.makedirs(d, exist_ok=True)
print('WORK_DIR   =', WORK_DIR)
print('DRIVE_ROOT =', DRIVE_ROOT)

In [ ]:
# Download from Zenodo (same URLs as the repo README)
ZENODO = 'https://zenodo.org/api/records/8196385/files/{}/content'
zips = {}
for name in ('HDFS_v1.zip', 'BGL.zip'):
    dest = os.path.join(WORK_DIR, name)
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f'[skip] {name} already downloaded')
    else:
        print(f'downloading {name} ...', flush=True)
        t0 = time.time()
        urllib.request.urlretrieve(ZENODO.format(name), dest)
        print(f'  done in {time.time() - t0:.0f}s, {os.path.getsize(dest) / 1e6:.0f} MB')
    zips[name] = dest

In [ ]:
# Extract (some Loghub zips nest a .tar.gz inside) and locate the files we need
def extract_all(zip_path, dest):
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dest)
    for tar in glob.glob(os.path.join(dest, '**', '*.tar.gz'), recursive=True):
        with tarfile.open(tar) as tf:
            tf.extractall(dest)

extract_dir = os.path.join(WORK_DIR, 'extracted')
os.makedirs(extract_dir, exist_ok=True)
for name, zp in zips.items():
    marker = os.path.join(extract_dir, '.' + name + '.done')
    if not os.path.exists(marker):
        print('extracting', name, flush=True)
        extract_all(zp, extract_dir)
        open(marker, 'w').close()

def largest(pattern):
    hits = [p for p in glob.glob(os.path.join(extract_dir, '**', pattern), recursive=True)
            if os.path.isfile(p)]
    assert hits, f'not found in archive: {pattern}'
    return max(hits, key=os.path.getsize)   # full dataset, not the 2k-line samples

hdfs_log = largest('HDFS*.log')
label_csv = largest('anomaly_label*')
bgl_log = largest('BGL*.log')
for p in (hdfs_log, label_csv, bgl_log):
    print(f'{os.path.getsize(p) / 1e6:9.1f} MB  {p}')

# Repo layout expected downstream (mirrors src/partition.py)
LAYOUT = {
    hdfs_log: 'data/HDFS/HDFS.log',
    label_csv: 'data/HDFS/preprocessed/anomaly_label.csv',
    bgl_log: 'data/BGL/BGL.log',
}

In [ ]:
# Persist to Drive in the repo layout
for src, rel in LAYOUT.items():
    dest = os.path.join(DRIVE_ROOT, rel)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest):
        print(f'copying -> {dest} ...', flush=True)
        shutil.copy2(src, dest)
    else:
        print(f'[skip] {dest} exists')

In [ ]:
# Validate against the numbers reported in the paper (section 4.3.2)
import re
import pandas as pd

blk_re = re.compile(r'blk_-?\d+')
n_lines, blocks, head = 0, set(), []
with open(os.path.join(DRIVE_ROOT, 'data/HDFS/HDFS.log'), encoding='utf-8', errors='replace') as f:
    for line in f:
        n_lines += 1
        if len(head) < 2:
            head.append(line)
        m = blk_re.search(line)
        if m:
            blocks.add(m.group())
print(''.join(head))
print(f'HDFS.log : {n_lines:,} lines, {len(blocks):,} blocks mentioned')
print('  paper  : 11,175,629 lines, 575,061 sessions')

lab = pd.read_csv(os.path.join(DRIVE_ROOT, 'data/HDFS/preprocessed/anomaly_label.csv'))
n_anom = int(lab['Label'].eq('Anomaly').sum())
print(f'labels   : {len(lab):,} blocks, {n_anom:,} anomalous ({n_anom / len(lab):.1%})')
print('  paper  : 575,061 blocks, 16,838 anomalous (2.9%)')

n_lines, n_alert, head = 0, 0, []
with open(os.path.join(DRIVE_ROOT, 'data/BGL/BGL.log'), encoding='utf-8', errors='replace') as f:
    for line in f:
        parts = line.split(None, 2)
        n_lines += 1
        if len(head) < 2:
            head.append(line)
        if parts[0] != '-':        # Loghub BGL: Label '-' = normal, anything else = alert
            n_alert += 1
print(''.join(head))
print(f'BGL.log  : {n_lines:,} lines, {n_alert:,} alert lines ({n_alert / n_lines:.1%})')
print('  paper  : 4,747,963 lines, 348,460 alert lines (7.3%)')

In [ ]:
# Data card: what is stored where (read by notebook 02)
card = {
    'source': 'Loghub (He et al., ISSRE 23 data track), Zenodo record 8196385',
    'urls': {n: ZENODO.format(n) for n in zips},
    'files': {rel: os.path.join(DRIVE_ROOT, rel) for rel in LAYOUT.values()},
    'created': time.strftime('%Y-%m-%d %H:%M:%S'),
}
with open(os.path.join(DRIVE_ROOT, 'data', 'data_card.json'), 'w') as f:
    json.dump(card, f, indent=2)

print(json.dumps(card, indent=2))

Done. Google Drive now holds (folder `LOG6309E/data`):

```
HDFS/HDFS.log                        (1.47 GB, 11,175,629 lines)
HDFS/preprocessed/anomaly_label.csv (575,061 block labels)
BGL/BGL.log                          (709 MB, 4,747,963 lines)
data_card.json
```

Next: run `notebooks/02_replicate_rq1.ipynb` — it stages this data into the cloned repo and runs
the full RQ1 pipeline (Drain parsing → sessions → MCV / TF-IDF (Text) → RF + LSTM).